# ipynb import and export

> Reading and writing dialogs as Jupyter notebooks

In [ ]:
#| default_exp ipynb

Dialogs are stored as Jupyter notebooks (`.ipynb` files), so any notebook tool can open them. This module handles the conversion:

| Message Type | Cell Type | Notes |
|-------------|-----------|-------|
| `note` | markdown | Direct mapping |
| `prompt` | code | Has `solveit_ai` metadata and a `%%prompt` first line; reply stored as an `is_ai_res` output |
| `code` | code | Outputs preserved as standard notebook outputs |
| `raw` | raw | Direct mapping |

**Writing** converts `Dialog` -> `.ipynb`, **Reading** does the reverse.

In [ ]:
#| export
from fastcore.utils import *
from fastcore.xtras import atomic_save
from fastcore.nbio import mk_cell,new_nb,dict2nb,read_nb,nb2str,repair_cell,repair_nb
import json
from contextlib import suppress
from base64 import b64encode,b64decode
from aidialog.dialog import *

In [ ]:
import random,os,nbformat
from nbformat.validator import NotebookValidationError
from tempfile import mkdtemp
from fastcore.test import *

In [ ]:
random.seed(7)
tstdir = Path(mkdtemp())
dlg = Dialog(name='dlg')
nt_msg = dlg.mk_message('A *test* dialog', msg_type=snote)
nt_msg.mk_attachment(b'not really a png', 'image/png')
code_msg = dlg.mk_message('1+1', msg_type=scode, output=code_output('2'))
ai_msg = dlg.mk_message('Add them.', msg_type=sprompt, output='The answer is **2**.')
raw_msg = dlg.mk_message('plain text', msg_type=sraw)
dlg


**dlg**


<details markdown='1'>

- A *test* dialog
- 1+1 ⇒ [{'output_type': 'execute_result', 'metadata': {}, 'data': …
- Add them. ⇒ [{'output_type': 'display_data', 'metadata': {'is_ai_res': …
- plain text

</details>

## Writing

Attachments look like this:
```js
  {
   "attachments": {
    "image.png": {
     "image/png": "iVBO...kJggg=="
    }
  },
```

In [ ]:
#| export
def att2dict(att): return {att.content_type: b64encode(att.data).decode('ascii') if isinstance(att.data, bytes) else att.data}

nbformat allows attachments only on markdown and raw cells, but a message of any type can carry them. rustygate solved this by storing a code cell's attachments in notebook metadata, under `metadata.attachments[cell_id]` (its docs call this the *home*), while presenting the ordinary cell shape through its API. Files must mean the same thing whoever wrote them, so the file boundary here speaks the same convention: writing homes code cells' attachments, reading brings them back, and between the boundaries every `Message` simply has `attachments`.

In [ ]:
#| export
def home_atts(nb):
    "Move code cells' attachments into the notebook-metadata home rustygate uses"
    home = nb['metadata'].setdefault('attachments', {})
    for c in nb['cells']:
        if c['cell_type']=='code' and (atts := c.pop('attachments', None)): home[c['id']] = atts
    if not home: nb['metadata'].pop('attachments', None)
    return nb

def unhome_atts(nb):
    "Reattach homed attachments onto their code cells: the inverse of `home_atts`"
    home = nb['metadata'].pop('attachments', {})
    for c in nb['cells']:
        if (atts := home.get(c['id'])): c['attachments'] = atts
    return nb

Prompts hold user input *and* AI response. A prompt serializes as a code cell: `solveit_ai: true` in metadata, a `%%prompt` first line, the question as the rest of the source, and the reply as a `display_data` output whose metadata carries `is_ai_res` — the same object `prompt_output` builds in memory, so the file stores the model's own shape. The magic line means a runner outside solveit fails cleanly (`UsageError`) instead of executing prose as Python, and a system that wants real behavior can define the magic. `cell2msg` strips it on the way back in.

In [ ]:
#| export
_out_meta_skip = {'__type'}

def _clean_out_meta(o):
    "A copy of output `o` without transient metadata; never mutates the live output"
    o = dict(o)
    if m := o.get('metadata'): o['metadata'] = {k:v for k,v in m.items() if k not in _out_meta_skip}
    return o

In [ ]:
#| export
@patch
def cell_meta(self:Message):
    "Metadata dict to write: `meta` plus demoted `meta_attrs` fields, falsy values omitted"
    meta = dict(self.meta)
    for a,k in self.meta_attrs.items():
        if (v := getattr(self, a, None)): meta[k] = v
        else: meta.pop(k, None)
    return meta

`cell_meta` assembles everything `to_cell` will write as cell metadata, and is the override point when a host's in-memory types differ from the file's: convert after `super()`, before serialization (solveit stores its UI flags as ints, but the ipynb schema types `collapsed`/`hide_input` as boolean).

In [ ]:
#| export
_prompt_magic = '%%prompt'

@patch
def to_cell(self:Message, version=2):
    "Convert message to a notebook cell"
    meta = self.cell_meta()
    src = self.source
    if self.msg_type==sprompt:
        meta['solveit_ai'] = True
        src = f'{_prompt_magic}\n{src}'
    outkw = {}
    if self.msg_type in (scode,sprompt) and self.output:
        outputs = self.output
        if version==1 and self.msg_type==scode: outputs = json.loads(outputs)
        outkw['outputs'] = [_clean_out_meta(o) for o in outputs]
    atts = {att.id: att2dict(att) for att in (self.attachments or [])}
    if atts: outkw['attachments'] = atts
    cell = mk_cell(src, self.cell_type, id=self.id, metadata=meta, **outkw)
    if repairs := repair_cell(cell): print('NB repair:', '; '.join(repairs))
    return cell

In [ ]:
om = Message('x', msg_type='code', output=[dict(output_type='execute_result', data={'text/plain':'42'}, metadata={'__type':'Safe'}, execution_count=1)])
test_eq(om.to_cell()['outputs'][0]['metadata'], {})  # transient metadata never reaches the file...
test_eq(om.output[0]['metadata'], {'__type':'Safe'})  # ...and serializing never mutates the live output (hosts read it after writes)

In [ ]:
test_eq(Message().to_cell()['metadata'],{})

`meta_attrs` is how a host teaches serialization about its fields: declare attribute → metadata key, and `cell_meta`/`cell2msg` demote/promote them (and the constructor promotes too, so `meta=` passed at creation reaches the attribute rather than being shadowed by it), with falsy values omitted from the file. Anything *not* declared still round-trips untouched inside `meta`:

In [ ]:
class NoteMsg(Message): meta_attrs = dict(bookmark='bookmark')
class NoteDlg(Dialog): msg_cls = NoteMsg

bookmark_cell = NoteMsg(bookmark=9).to_cell()
test_eq(bookmark_cell['metadata']['bookmark'], 9)
test_eq(NoteMsg().to_cell()['metadata'], {})  # default/absent values aren't written

In [ ]:
test_eq(NoteMsg('x', meta=dict(bookmark=7)).bookmark, 7)  # constructor meta promotes like a file read
test_eq(NoteMsg('x', meta=dict(bookmark=7)).to_cell()['metadata'], {'bookmark': 7})  # ...so it round-trips to the file

When a host's in-memory type differs from what the file should carry (the ipynb schema types some keys), it converts in a `cell_meta` override:

In [ ]:
class FlagMsg(Message):
    meta_attrs = dict(collapsed='collapsed')
    def cell_meta(self): return {k: bool(v) for k,v in super().cell_meta().items()}

assert FlagMsg(collapsed=1).to_cell()['metadata']['collapsed'] is True

In [ ]:
pr_msg = dlg.mk_message('What is 2+2?', output='The answer is 4.', msg_type='prompt')
pr_cell = pr_msg.to_cell()
test_eq((pr_cell['cell_type'], pr_cell['metadata']['solveit_ai']), ('code', True))
test_eq(pr_cell['source'], '%%prompt\nWhat is 2+2?')
test_eq(pr_cell['outputs'], prompt_output('The answer is 4.'))

pr_empty = dlg.mk_message('Hello?', output='', msg_type='prompt')
test_eq(pr_empty.to_cell()['outputs'], [])
pr_cell


In [ ]:
#| export
def get_ipynb(dlg:Dialog, version=2, msgs=None):
    "Notebook object for `dlg`; `msgs` defaults to all its messages"
    cells = [m.to_cell(version=version) for m in (dlg.messages if msgs is None else msgs)]
    nb = new_nb(cells=cells, meta=dict(dlg.meta))
    home_atts(nb)
    if repairs := repair_nb(nb): print('NB repair:', '; '.join(repairs))
    return nb


In [ ]:
#| export
def safe_mtime(p):
    "mtime of `p`, its symlink if the target is missing, or None if `p` vanished (e.g. an atomic-save temp file)"
    with suppress(FileNotFoundError): return p.stat().st_mtime
    with suppress(FileNotFoundError): return p.stat(follow_symlinks=False).st_mtime

In [ ]:
#| export
def write_ipynb(dlg:Dialog, fname=None, version=2, msgs=None, **kwargs):
    "Write `dlg` as a notebook, or return the JSON string if `fname` is None; `kwargs` (e.g. `uid`/`gid`) pass to `atomic_save`"
    res = nb2str(get_ipynb(dlg, version=version, msgs=msgs))
    if not fname: return res
    fname = Path(fname).expanduser()
    with atomic_save(fname, mode='w', encoding='utf-8', **kwargs) as f: f.write(res)
    dlg.mtime_ = safe_mtime(fname)

In [ ]:
write_ipynb(dlg, fname=tstdir/'dlg.ipynb', version=2)

In [ ]:
#| export
@patch
def write(self:Dialog, base_path, version=2, msgs=None, **kwargs):
    write_ipynb(self, Path(base_path).expanduser()/f'{self.name}.ipynb', version=version, msgs=msgs, **kwargs)

In [ ]:
dlg.write(tstdir)

In [ ]:
#| export
def ipynb_cells(path, nm, prefix=None, suffix=None):
    tmpl = Path(path).expanduser()/f'{nm}.ipynb'
    if not tmpl.exists(): return []
    try: nb = read_nb(tmpl)
    except json.JSONDecodeError: return []
    if repairs := repair_nb(nb): print('NB repair:', '; '.join(repairs))
    return listify(prefix) + nb.cells + listify(suffix)

In [ ]:
test_eq(len(ipynb_cells(tstdir, dlg.name)), len(dlg.messages))
ipynb_cells(tstdir, dlg.name)[0]

```python
{ 'attachments': { '2530bb1d-6d13-4cde-8623-7b2ed91e3f72': { 'image/png': 'bm90IHJlYWxseSBhIHBuZw=='}},
  'cell_type': 'markdown',
  'id': 'a54dca18',
  'idx_': 0,
  'lang_': 'python',
  'metadata': {},
  'source': 'A *test* dialog'}
```

## Reading

In [ ]:
#| export
def dict2att(att_id, att_data):
    "Convert attachment dict to Attachment object"
    content_type, data = first(att_data.items())
    if isinstance(data, str): data = b64decode(data)
    return Attachment(data, content_type, att_id)

def _output_from_cell(cell):
    if cell.cell_type!='code': return ''
    return getattr(cell, 'outputs', [])

In [ ]:
#| export
@patch
def cell2msg(self:Dialog, cell):
    "Convert single notebook cell to message object"
    meta = dict(cell.metadata)
    kwargs = {a: meta.pop(k) for a,k in self.msg_cls.meta_attrs.items() if k in meta}
    content = cell.get('source', '')
    msg_type = sprompt if meta.pop('solveit_ai', 0) else snote if cell.cell_type=='markdown' else cell.cell_type
    if msg_type==sprompt and content.split('\n', 1)[0].strip()==_prompt_magic:
        content = content.split('\n', 1)[1] if '\n' in content else ''
    output = '' if msg_type in (snote,sraw) else _output_from_cell(cell)
    atts = [dict2att(att_id, att_data) for att_id, att_data in cell.get('attachments', {}).items()]
    id = getattr(cell, 'id', None) or rtoken_hex(4)
    return self.msg_cls(content, id=id, output=output, msg_type=msg_type, dlg=self, attachments=atts, meta=meta, **kwargs)

In [ ]:
back = NoteDlg(name='t').cell2msg(bookmark_cell)
test_eq(back.bookmark, 9)
test_eq(back.meta, {})  # promoted out of `meta` into the attribute

In [ ]:
# Test roundtrip prompts
pr_back = dlg.cell2msg(pr_cell)
test_eq(pr_back.content, 'What is 2+2?')
test_eq(pr_back.msg_type, 'prompt')
test_eq(pr_back.ai_res, 'The answer is 4.')

In [ ]:
#| export
@patch
def from_cells(self:Dialog, cells):
    self.messages = Msgs(cells).map(self.cell2msg)
    return self

In [ ]:
#| export
def reads_ipynb(txt, cls=Dialog, name='dialog', verbose=False):
    "Read a dialog from notebook JSON string `txt`, constructing via `cls`"
    nb = json.loads(txt)
    if (repairs := repair_nb(nb)) and verbose: print('NB repair:', '; '.join(repairs))
    unhome_atts(nb)
    nb = dict2nb(nb)
    return cls(name=name, meta=dict(nb.get('metadata', {}))).from_cells(nb.cells)

In [ ]:
#| export
def read_ipynb(fname, cls=Dialog, name=None, verbose=False):
    "Read a dialog from notebook file `fname` (`.ipynb` added if missing), constructing via `cls`; `name` defaults to the file stem"
    f = Path(fname).expanduser()
    if f.suffix != '.ipynb': f = f.with_suffix('.ipynb')
    if not f.exists(): return print(f,'does not exist')
    try: res = reads_ipynb(f.read_text(encoding='utf-8'), cls, name or f.stem, verbose=verbose)
    except (json.JSONDecodeError, PermissionError): return
    res.path_ = f
    res.mtime_ = safe_mtime(f)
    return res

In [ ]:
dlg = read_ipynb(tstdir/'dlg')
dlg


**dlg**


<details markdown='1'>

- A *test* dialog
- 1+1 ⇒ [{'data': {'text/plain': '2'}, 'execution_count': 1, 'metad…
- Add them. ⇒ [{'output_type': 'display_data', 'metadata': {'is_ai_res': …
- plain text
- What is 2+2? ⇒ [{'output_type': 'display_data', 'metadata': {'is_ai_res': …
- Hello?

</details>

In [ ]:
s = (tstdir/'dlg.ipynb').read_text()
dlg2 = reads_ipynb(s)
test_eq(dlg2.name, 'dialog')
test_eq(write_ipynb(dlg2), s)
with expect_fail(json.JSONDecodeError): reads_ipynb('not json')

A code message's attachments take the homed path both ways: the written cell stays schema-valid, `metadata.attachments` carries the bytes under the cell's id, and reading brings them back onto the message. The canonical-form fixed point holds through the home too.

In [ ]:
hm = Dialog(name='home')
cm = hm.mk_message('1+1', msg_type=scode, output=code_output('2'))
cm.mk_attachment(b'not really a png', 'image/png')
hs = write_ipynb(hm)
nbformat.validate(nbformat.reads(hs, as_version=4))
hnb = json.loads(hs)
assert 'attachments' not in hnb['cells'][0]
test_eq(reads_ipynb(hs).messages[0].attachments[0].data, b'not really a png')
test_eq(write_ipynb(reads_ipynb(hs)), hs)
hnb['metadata']['attachments']

In [ ]:
#| export
@patch
def save(self:Dialog, fname=None):
    "Write back to `fname`, or to the `path_` stamped by `read_ipynb`"
    fname = fname or self.path_
    if not fname: raise ValueError('no fname passed, and no `path_` stamped by read_ipynb')
    write_ipynb(self, fname)

`read_ipynb` stamps the source path as `path_` (the trailing-underscore working-attribute convention, staying clear of hosts' own `path` properties), so load-edit-save needs no path threading. It also stamps the file's `safe_mtime` as `mtime_` (as does a `write_ipynb` given a `fname`), so a host can tell its own writes from external ones by comparing the file's current mtime against the stamp. Both default to `None` on the class, so a dialog that has never touched disk answers rather than raising. Saving an unedited dialog is a byte no-op:

In [ ]:
before = (tstdir/'dlg.ipynb').read_text()
dlg.save()
test_eq((tstdir/'dlg.ipynb').read_text(), before)
test_eq(dlg.mtime_, dlg.path_.stat().st_mtime)
assert isinstance(dlg.messages, Msgs)
unread = Dialog(name='unread')
test_eq((unread.path_, unread.mtime_), (None,None))
with expect_fail(ValueError, 'path_'): unread.save()

### Metadata preservation

The base class itself names just two fields, shared by every host — `skipped` (hidden from AI context) and `pinned` (kept through context eviction), under solveit's literal keys so files mean the same thing everywhere. Everything else in cell metadata that a `meta_attrs` declaration doesn't claim rides verbatim in `Message.meta`, and notebook-level metadata rides in `Dialog.meta` the same way. So a file written by any host survives a read/write round trip through the plain classes with every annotation intact:

In [ ]:
m0 = dlg.messages[0]
m0.meta['my_app_flag'] = dict(level=3)
dlg.meta['my_app'] = dict(version=1)
dlg.write(tstdir)
dlg = read_ipynb(tstdir/'dlg')
test_eq(dlg.messages[0].meta['my_app_flag'], dict(level=3))
test_eq(dlg.meta['my_app'], dict(version=1))

`skipped` and `pinned` ride the same mechanism as base-class fields: set them as attributes, and they demote to the solveit-compatible metadata keys on write (falsy omitted, so pristine messages add nothing to the file) and promote back to attributes on read.

In [ ]:
m0 = dlg.messages[0]
m0.skipped = 1
m0.pinned = True
dlg.write(tstdir)
rt = read_ipynb(tstdir/'dlg')
test_eq(rt.messages[0].skipped, 1)                  # promoted back as a real attribute
test_eq(rt.messages[0].pinned, True)
assert 'skipped' not in rt.messages[0].meta         # claimed by meta_attrs, so not left in meta
test_eq(rt.messages[1].skipped, 0)                  # untouched message: class default, and
assert 'skipped' not in rt.messages[1].cell_meta()  # ...falsy means nothing written to the file

### Round-trip fidelity

`write_ipynb` normalizes as it writes. A cell-level execution count drops to `null`, for instance, so a freshly-run notebook loses its counts on the first save. That written form is canonical, and read/write has it as a fixed point. Repeated open/save never changes a byte, and once a file is canonical, any byte change means a real content change. Repo notebooks are more varied than constructed examples, so each one round-trips through its canonical form:

In [ ]:
for p in sorted(Path('.').glob('0*.ipynb')):
    s = write_ipynb(read_ipynb(str(p)))
    test_eq(write_ipynb(reads_ipynb(s)), s)

Schema-invalid files (hand-edited internals happen: this stray `outputs` key on a markdown cell is from a real file) aren't preserved but healed, because `Message` has no slot for the invalid part. The content survives; the invalidity doesn't:

In [ ]:
bad_cells = [dict(cell_type='markdown', id='aaaa1111', metadata={}, source='hi', outputs=[])]
(tstdir/'bad.ipynb').write_text(json.dumps(dict(nbformat=4, nbformat_minor=5, metadata={}, cells=bad_cells)))
with expect_fail(NotebookValidationError): nbformat.validate(nbformat.read(tstdir/'bad.ipynb', as_version=4))
healed = write_ipynb(read_ipynb(tstdir/'bad'))
nbformat.validate(nbformat.reads(healed, as_version=4))
assert 'hi' in healed

NB repair: cell aaaa1111: removed outputs


## Converting old prompts

`conv_old_prompts` rewrites a notebook's markdown-form prompt cells to the code-cell form, in place, and returns the ids it changed — so a host can convert a file the first time it opens it, and write back only when something changed. Converted cells no longer match the predicate, so it is idempotent by construction. A retype makes any prompt attachments illegal where they sit, so it finishes with `home_atts`.

In [ ]:
#| export
_reply_sep = "\n\n##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->\n\n"

def conv_old_prompts(nb):
    "Rewrite old markdown-form prompt cells to the code-cell form, in place; returns the changed cell ids"
    changed = []
    for c in nb['cells']:
        if c['cell_type']!='markdown' or not c['metadata'].get('solveit_ai'): continue
        if isinstance(c['source'], list): c['source'] = ''.join(c['source'])
        content,*reply = c['source'].split(_reply_sep)
        c['cell_type'] = 'code'
        c['source'] = f'{_prompt_magic}\n{content}'
        c['outputs'] = prompt_output(reply[0]) if reply else []
        c['execution_count'] = None
        changed.append(c['id'])
    if changed: home_atts(nb)
    return changed

Old files are the only place the markdown form still exists: one markdown cell with `solveit_ai` metadata, holding the question and reply joined by a separator line. The lesson states one literally — the separator's last appearance outside the converter. Conversion keeps every part of the message and leaves a schema-valid file that reads back identically:

In [ ]:
onb = dict(nbformat=4, nbformat_minor=5, metadata={}, cells=[dict(
    cell_type='markdown', id='opr1', metadata=dict(solveit_ai=True),
    source='What is 7*6?' + _reply_sep + '**42**.',
    attachments={'att1': {'image/png': 'bm90IHJlYWxseSBhIHBuZw=='}})])
test_eq(conv_old_prompts(onb), ['opr1'])
ocell = onb['cells'][0]
test_eq((ocell['cell_type'], ocell['source']), ('code', '%%prompt\nWhat is 7*6?'))
assert 'attachments' not in ocell                # the retype homed the attachment
nbformat.validate(nbformat.from_dict(onb))
test_eq(conv_old_prompts(onb), [])               # converted cells no longer match: idempotent
back = reads_ipynb(json.dumps(onb)).messages[0]
test_eq((back.content, back.ai_res), ('What is 7*6?', '**42**.'))
test_eq(back.attachments[0].data, b'not really a png')
back.ai_res

## export -

In [ ]:
#|hide
#|eval: false
import nbdev; nbdev.nbdev_export()